# 로지스틱 회귀 실습 — 간단한 이진 분류

**로지스틱 회귀(Logistic Regression)** 는 이름에 '회귀'가 들어가지만 실제로는 **분류(Classification)** 알고리즘입니다. 선형 회귀처럼 `w·x + b` 를 계산한 뒤, **시그모이드(sigmoid)** 함수로 0~1 사이의 **확률**로 변환해 클래스를 예측합니다.

여기서는 2개의 특성을 가진 점들을 두 그룹(0/1)으로 나누는 가장 간단한 이진 분류를 만들고, 모델이 학습한 **결정 경계(decision boundary)** 를 시각화합니다.

## 목차
1. 라이브러리 임포트
2. 데이터 생성
3. 학습/테스트 분할
4. 모델 학습
5. 예측 및 성능 평가
6. 시그모이드(로지스틱) 함수 살펴보기
7. 결정 경계 시각화

## 1. 라이브러리 임포트

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 그래프 한글 깨짐 방지 (Ubuntu: NanumGothic)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)  # 재현성

## 2. 데이터 생성

`make_classification` 으로 2개 특성을 가진 300개 점을 만듭니다. 두 클래스가 약간 겹치도록 설정해(`class_sep`, `flip_y`) 100% 완벽하지 않은 현실적인 분류 문제로 만듭니다.

In [ ]:
X, y = make_classification(
    n_samples=300, n_features=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=0.9, flip_y=0.05,
    random_state=42
)
print('X shape:', X.shape, '/ 클래스 분포:', np.bincount(y))

plt.figure(figsize=(7, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', alpha=0.6, edgecolors='k')
plt.xlabel('특성 1'); plt.ylabel('특성 2')
plt.title('생성된 이진 분류 데이터 (빨강=1, 파랑=0)')
plt.grid(True, alpha=0.3)
plt.show()

## 3. 학습/테스트 분할

학습 80% / 테스트 20%로 나눕니다. `stratify=y` 로 두 클래스 비율을 유지합니다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('학습 샘플:', len(X_train), '/ 테스트 샘플:', len(X_test))

## 4. 모델 학습

`LogisticRegression` 을 만들고 `fit()` 으로 학습합니다. 학습된 **계수(coef_)** 와 **절편(intercept_)** 이 결정 경계 직선 `w1·x1 + w2·x2 + b = 0` 을 정의합니다.

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

print('계수(coef_)   :', model.coef_.round(3))
print('절편(intercept_):', model.intercept_.round(3))

## 5. 예측 및 성능 평가

테스트셋으로 예측해 **정확도**, 분류 리포트(정밀도/재현율/F1), 혼동 행렬을 확인합니다. `predict_proba` 로는 각 샘플이 클래스 1일 **확률**도 볼 수 있습니다.

In [ ]:
y_pred = model.predict(X_test)

print(f'정확도: {accuracy_score(y_test, y_pred):.3f}\n')
print(classification_report(y_test, y_pred))
print('혼동 행렬:')
print(confusion_matrix(y_test, y_pred))

# 앞 5개 샘플의 클래스 1 확률
proba = model.predict_proba(X_test)[:5, 1]
print('\n앞 5개 샘플의 클래스 1 예측 확률:', proba.round(3))

## 6. 시그모이드(로지스틱) 함수 살펴보기

로지스틱 회귀의 핵심은 선형 점수 `z = w·x + b` 를 **시그모이드 함수** 로 0~1 확률로 바꾸는 것입니다.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

- `z = 0` → 확률 **0.5** (결정 경계)
- `z` 가 커질수록 → 확률이 **1**(클래스 1)에 가까워짐
- `z` 가 작아질수록 → 확률이 **0**(클래스 0)에 가까워짐

왼쪽은 순수 시그모이드 곡선, 오른쪽은 우리 데이터의 결정 점수에 시그모이드를 적용한 실제 S-커브입니다.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# 모델의 결정 점수 z = w·x + b  (테스트 데이터)
z_test = X_test @ model.coef_[0] + model.intercept_[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (1) 순수 시그모이드 함수
zz = np.linspace(-8, 8, 200)
axes[0].plot(zz, sigmoid(zz), color='purple', linewidth=2)
axes[0].axhline(0.5, color='gray', ls='--', alpha=0.7)
axes[0].axvline(0, color='gray', ls='--', alpha=0.7)
axes[0].set_xlabel('z = w·x + b'); axes[0].set_ylabel('확률 σ(z)')
axes[0].set_title('시그모이드 함수  σ(z) = 1 / (1 + e^(-z))')
axes[0].grid(True, alpha=0.3)

# (2) 실제 데이터: 결정 점수 vs 클래스 1 확률
order = np.argsort(z_test)
axes[1].scatter(z_test, y_test, c=y_test, cmap='bwr', alpha=0.6,
                edgecolors='k', label='실제 클래스 (0/1)')
axes[1].plot(z_test[order], sigmoid(z_test[order]), color='purple',
             linewidth=2, label='예측 확률 σ(z)')
axes[1].axhline(0.5, color='gray', ls='--', alpha=0.7)
axes[1].set_xlabel('결정 점수 z = w·x + b'); axes[1].set_ylabel('클래스 1 확률')
axes[1].set_title('데이터에 적용된 시그모이드 (z↑ → 클래스 1)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 결정 경계 시각화

입력 평면 전체에 대해 모델이 예측하는 클래스를 배경색으로 칠하면, 두 클래스를 가르는 **결정 경계**가 직선으로 나타납니다. 로지스틱 회귀는 (선형) 직선 경계를 학습하므로, 겹치는 영역의 점은 일부 오분류됩니다.

In [ ]:
# 격자 생성
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

# 격자 각 점의 예측
Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='bwr')                 # 예측 영역
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='bwr',   # 테스트 점
            alpha=0.9, edgecolors='k')
plt.xlabel('특성 1'); plt.ylabel('특성 2')
plt.title(f'로지스틱 회귀 결정 경계 (정확도={accuracy_score(y_test, y_pred):.3f})')
plt.grid(True, alpha=0.3)
plt.show()

## 정리

- **로지스틱 회귀** 는 `w·x + b` 를 **시그모이드** 로 확률(0~1)로 바꿔 분류하는 모델입니다.
- scikit-learn에서는 `LogisticRegression().fit(X, y)` 로 학습하고, `predict`(클래스)·`predict_proba`(확률) 로 예측합니다.
- 결정 경계는 **직선**(선형)이라, 겹치는 데이터는 일부 오분류됩니다.
- 더 해보기: 3개 이상 클래스(다항 로지스틱), 비선형 경계가 필요하면 **SVM(RBF)** / **결정 트리** / **신경망**, 과적합 제어를 위한 규제(`C` 값) 조정을 시도해 보세요.

> 참고: 회귀(연속값 예측)는 [simple_regression_example.ipynb](simple_regression_example.ipynb) 를, 같은 데이터 분류 비교는 [Decision_Tree.ipynb](Decision_Tree.ipynb) · [SVM.ipynb](SVM.ipynb) 를 함께 보세요.